# HSBC Statements Processing

Extract and analyze transaction data from HSBC PDF statements.

In [21]:
import { saveJson, readJson } from "./utils.ts";
import {PDFParse, } from "npm:pdf-parse";
import { readdir, readFile } from "node:fs/promises";
import {join} from "node:path"
const statementsDir = "./data/hsbcStatements";

// Get all PDF files
const pdfFiles: string[] = [];
const dir = await readdir(statementsDir)
for await (const entry of dir) {
  if (entry.endsWith(".pdf")) {
    pdfFiles.push(entry);
  }
}
pdfFiles.sort();
console.log(`Found ${pdfFiles.length} PDF statements:`, pdfFiles);

Found 12 PDF statements: [
  "2025-01-16_Statement.pdf",
  "2025-02-16_Statement.pdf",
  "2025-03-16_Statement.pdf",
  "2025-04-16_Statement.pdf",
  "2025-05-16_Statement.pdf",
  "2025-06-16_Statement.pdf",
  "2025-07-16_Statement.pdf",
  "2025-08-16_Statement.pdf",
  "2025-09-16_Statement.pdf",
  "2025-10-16_Statement.pdf",
  "2025-11-16_Statement.pdf",
  "2025-12-16_Statement.pdf"
]


In [22]:
import * as pdfjsLib from "npm:pdfjs-dist";
import pl from 'npm:nodejs-polars';
import { displayTable } from "./utils.ts";

interface pdfTextRecord {
  text: string
  x: number
  y: number
  width: number
  height: number
  page: number
}
async function extract(pdfPath: string): Promise<pdfTextRecord[]> {

const dataBuffer = await readFile(pdfPath);
const pdfData = new Uint8Array(dataBuffer);
const pdfDoc = await pdfjsLib.getDocument({ data: pdfData, disableWorker: true} as any).promise;
const textRecords: pdfTextRecord[] = []
for (let i = 1; i < pdfDoc.numPages; i++) {
  const page = await pdfDoc.getPage(i);
  const textContent = await page.getTextContent();

  for (const item of textContent.items) {
    const unTypedItem = item as any
    const y = Math.round(unTypedItem.transform[5]); // round y to group lines
    const x = Math.round(unTypedItem.transform[4]);
    const textRecord = {
      text: unTypedItem.str,
      x: x,
      y: y,
      width: unTypedItem.width,
      height: unTypedItem.height,
      page: i
    }

    textRecords.push(textRecord)
  }
}
return textRecords
}

const samplePdf = pdfFiles[0];
const pdfPath = join(statementsDir, samplePdf);
const textRecords = await extract(pdfPath)
const df = pl.DataFrame(textRecords)
displayTable(df)

displayTable(df.where(pl.col("text").str.contains("1,974.03")))

{ height: 512, width: 6 }
┌───────┬─────────────────────────────────────────────┬─────┬─────┬────────────────────┬────────┬──────┐
│ (idx) │ text                                        │ x   │ y   │ width              │ height │ page │
├───────┼─────────────────────────────────────────────┼─────┼─────┼────────────────────┼────────┼──────┤
│     0 │ "see reverse for call times"                │ 462 │ 753 │ 79.52800000000002  │      8 │    1 │
│     1 │ "used by deaf or speech impaired customers" │ 394 │ 732 │ 137.74400000000003 │      8 │    1 │
│     2 │ "A"                                         │  53 │ 376 │ 5.776              │      8 │    1 │
│     3 │ "."                                         │ 313 │ 366 │ 2                  │      8 │    1 │
│     4 │ "18 Dec 24"                                 │  53 │ 356 │ 32.879999999999995 │      8 │    1 │
│     5 │ " "                                         │  86 │ 356 │ 26.819999999999993 │      0 │    1 │
│     6 │ ")))"              

In [23]:



const dfWithXCount = df
  .withColumn(pl.lit(1).alias("count_x"))
  .groupBy("x")
  .agg(pl.count("count_x").sum())
  .sort("count_x", true)

displayTable(dfWithXCount)


const dfWithXRightCount = df.
  withColumn(pl.col("x").plus("width").alias("x_right"))
  .withColumn(pl.lit(1).alias("count_x"))
  .groupBy("x_right")
  .agg(pl.count("count_x").sum())
  .sort("count_x", true)
  .where(pl.col("x_right").gt(390))

displayTable(dfWithXRightCount)



{ height: 116, width: 2 }
┌───────┬─────┬─────────┐
│ (idx) │ x   │ count_x │
├───────┼─────┼─────────┤
│     0 │ 139 │      62 │
│     1 │  52 │      39 │
│     2 │ 140 │      28 │
│     3 │ 112 │      27 │
│     4 │  53 │      16 │
│     5 │ 371 │      16 │
│     6 │ 125 │      14 │
│     7 │ 529 │      13 │
│     8 │ 113 │      13 │
│     9 │ 376 │      11 │
└───────┴─────┴─────────┘
{ height: 84, width: 2 }
┌───────┬───────────────────┬─────────┐
│ (idx) │ x_right           │ count_x │
├───────┼───────────────────┼─────────┤
│     0 │ 551               │      15 │
│     1 │ 550               │       8 │
│     2 │ 529.3             │       5 │
│     3 │ 521.9000000000001 │       3 │
│     4 │ 521.4000000000001 │       3 │
│     5 │ 538.88            │       3 │
│     6 │ 545.77            │       3 │
│     7 │ 546.93            │       3 │
│     8 │ 545.11            │       3 │
│     9 │ 438               │       3 │
└───────┴───────────────────┴─────────┘


In [24]:
const dfPaymentDetails = df.filter(
pl.col("x").minus(139).abs().ltEq(3)
)
// .withColumn(pl.lit(1).alias("count_x"))
// .groupBy("x")
// .agg(pl.col("count_x").sum())
console.log("payment details: ")
displayTable(dfPaymentDetails)


const dfDates = df.filter(
  pl.col("x").minus(53).abs().ltEq(3)
)
// .withColumn(pl.lit(1).alias("count_x"))
// .groupBy("x")
// .agg(pl.col("count_x").sum())
console.log("Dates")
displayTable(dfDates)


const dfPaidOut = df.
withColumn(pl.col("x").plus("width").alias("x_right"))
.filter(
  pl.col("x_right").minus(389).abs().ltEq(3)

)
// .withColumn(pl.lit(1).alias("count"))
// .groupBy("x_right")
// .agg(pl.col("count").sum())
// .sort("count", true)
console.log("paid out")
displayTable(dfPaidOut)

const dfPaidIn = df.
withColumn(pl.col("x").plus("width").alias("x_right"))
.filter(
  pl.col("x_right").minus(470).abs().ltEq(3)

)
// .withColumn(pl.lit(1).alias("count"))
// .groupBy("x_right")
// .agg(pl.col("count").sum())
// .sort("count", true)
console.log("paid in")
displayTable(dfPaidIn)


payment details: 
{ height: 90, width: 6 }
┌───────┬──────────────────────┬─────┬─────┬────────────────────┬────────┬──────┐
│ (idx) │ text                 │ x   │ y   │ width              │ height │ page │
├───────┼──────────────────────┼─────┼─────┼────────────────────┼────────┼──────┤
│     0 │ "WM MORRISONS STORE" │ 140 │ 356 │ 90.67200000000003  │      8 │    1 │
│     1 │ "BOLTON"             │ 140 │ 344 │ 32.44              │      8 │    1 │
│     2 │ "amazon.co.uk*M36YI" │ 140 │ 333 │ 71.98400000000001  │      8 │    1 │
│     3 │ "353-12477661"       │ 140 │ 321 │ 46.664             │      8 │    1 │
│     4 │ "UBER"               │ 140 │ 310 │ 21.336             │      8 │    1 │
│     5 │ "HELP.UBER.COM"      │ 140 │ 299 │ 63.559999999999995 │      8 │    1 │
│     6 │ "UBER"               │ 140 │ 287 │ 21.336             │      8 │    1 │
│     7 │ "HELP.UBER.COM"      │ 140 │ 276 │ 63.559999999999995 │      8 │    1 │
│     8 │ "GREGGS PLC"         │ 140 │ 264 │ 48.672    

In [30]:

interface BankTransaction {
  date: string,
  type: string,
  details: string,
  amount: number
}


export function isNumberString(str: string): boolean {
  if (str.replace(",", "").trim() === "") return false; // EUROPEANS BEWARE
  return Number.isFinite(Number(str.replace(",", "")));
}


function isRecordDate(record: pdfTextRecord) {
  // could also do date type check but cba
  const text = record.text
  const xAlign = 53
  if (text.split(" ").length !== 3) return false
  const [day, month, year]= text.split(" ")
  if (!isNumberString(day) || ! isNumberString(year)) return false
  if (Math.abs(record.x - xAlign) > 2) return false
  return true
}

function isRecordType(record: pdfTextRecord) {
  const xAlign = 113
  if (record.text.length > 3) return false
  if (Math.abs(record.x - xAlign) > 2) return false
  return true
}

function isRecordDetail(record: pdfTextRecord) {
  const xAlign = 140
  if (record.text.replace(" ", "").length === 0) return false
  if (Math.abs(record.x - xAlign) > 2) return false
  return true
}

function isRecordPaidOut(record: pdfTextRecord) {
  const xRight = record.x + record.width
  const xRightAlign = 390
  if (!isNumberString(record.text)) return false
  if (Math.abs(xRight - xRightAlign) > 4) return false
  return true
}

function isRecordPaidIn(record: pdfTextRecord) {

  const xRight = record.x + record.width
  const xRightAlign = 470
  if (!isNumberString(record.text)) return false
  if (Math.abs(xRight - xRightAlign) > 2) return false
  return true
}

// very stateful 😋
function transform(textRecords: pdfTextRecord[]) {
  const sortedRecords = textRecords.sort((recordA, recordB) => {
    const lowerPage = recordA.page < recordB.page
    const lowerY = recordA.y < recordB.y
    return Number(lowerPage) + Number(lowerY)
  })
  const bankTransactions: BankTransaction[] = []
  const emptyRecord = {
    date: "",
    type: "",
    details: "",
    amount: 0
  }
  const currentRecord: BankTransaction = { ...emptyRecord }
  type searchStates = "DATE_OR_TYPE" | "DETAIL_OR_AMOUNT"
  let searchingFor: searchStates = "DATE_OR_TYPE"
  sortedRecords.forEach((record: pdfTextRecord) => {
    if ((searchingFor === "DATE_OR_TYPE") && isRecordDate(record)) {
      currentRecord.date = record.text
      searchingFor = "DATE_OR_TYPE"
      return
    }
    if (searchingFor === "DATE_OR_TYPE" && isRecordType(record)) {
      currentRecord.type = record.text
      searchingFor = "DETAIL_OR_AMOUNT"
      return
    }
    if (searchingFor === "DETAIL_OR_AMOUNT" && isRecordDetail(record)) {
      currentRecord.details += " " + record.text
      searchingFor = "DETAIL_OR_AMOUNT"
      return
    }

    if (searchingFor === "DETAIL_OR_AMOUNT") {
      if (isRecordPaidOut(record)) {
        currentRecord.amount = -1 * Number(record.text.replace(",", ""))
        bankTransactions.push({ ...currentRecord })
        currentRecord.details = ""
        searchingFor = "DATE_OR_TYPE"
        return
      }

      if (isRecordPaidIn(record)) {

        currentRecord.amount = Number(record.text.replace(",", ""))

        bankTransactions.push({ ...currentRecord })
        currentRecord.details = ""
        searchingFor = "DATE_OR_TYPE"
        return
      }
    }
  })
  return bankTransactions
}
// find date
// find type
// find text
// find ammount
// console.log(textRecords.filter(record => record.text.includes("1,974")))
const bankTransactions = transform(textRecords).map(row => {
  return {
    ...row,
    details: row.details.slice(0, 3) + "***"
  }
})
console.table(bankTransactions.slice(0, 40))


┌───────┬─────────────┬───────┬──────────┬─────────┐
│ (idx) │ date        │ type  │ details  │ amount  │
├───────┼─────────────┼───────┼──────────┼─────────┤
│     0 │ "18 Dec 24" │ ")))" │ " WM***" │ -6.25   │
│     1 │ "18 Dec 24" │ "VIS" │ " am***" │ -4.49   │
│     2 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.91   │
│     3 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.98   │
│     4 │ "20 Dec 24" │ ")))" │ " GR***" │ -2      │
│     5 │ "20 Dec 24" │ ")))" │ " WM***" │ -9.05   │
│     6 │ "23 Dec 24" │ "VIS" │ " WW***" │ -42.17  │
│     7 │ "23 Dec 24" │ "VIS" │ " am***" │ -5.99   │
│     8 │ "23 Dec 24" │ ")))" │ " NY***" │ -2.2    │
│     9 │ "23 Dec 24" │ ")))" │ " WM***" │ -5      │
│    10 │ "23 Dec 24" │ "VIS" │ " am***" │ -3.49   │
│    11 │ "24 Dec 24" │ "VIS" │ " WM***" │ -44.23  │
│    12 │ "24 Dec 24" │ ")))" │ " NY***" │ -2.2    │
│    13 │ "25 Dec 24" │ "BP"  │ " An***" │ -50     │
│    14 │ "27 Dec 24" │ "DD"  │ " TH***" │ -24.99  │
│    15 │ "27 Dec 24" │ "VIS" │ " Am***" │ -35

In [26]:
Number("1,974.03") // bruh

NaN

In [27]:
import { maskColumn } from "./utils.ts";

const bankDf = pl.DataFrame(bankTransactions)

const groupedByDetail = bankDf
.groupBy("details")
.agg(pl.col("amount").sum())
.sort("amount")
displayTable(maskColumn(bankDf, "details"))
displayTable(maskColumn(groupedByDetail, "details"))

{ height: 40, width: 4 }
┌───────┬─────────────┬───────┬──────────┬────────┐
│ (idx) │ date        │ type  │ details  │ amount │
├───────┼─────────────┼───────┼──────────┼────────┤
│     0 │ "18 Dec 24" │ ")))" │ " WM***" │ -6.25  │
│     1 │ "18 Dec 24" │ "VIS" │ " am***" │ -4.49  │
│     2 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.91  │
│     3 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.98  │
│     4 │ "20 Dec 24" │ ")))" │ " GR***" │ -2     │
│     5 │ "20 Dec 24" │ ")))" │ " WM***" │ -9.05  │
│     6 │ "23 Dec 24" │ "VIS" │ " WW***" │ -42.17 │
│     7 │ "23 Dec 24" │ "VIS" │ " am***" │ -5.99  │
│     8 │ "23 Dec 24" │ ")))" │ " NY***" │ -2.2   │
│     9 │ "23 Dec 24" │ ")))" │ " WM***" │ -5     │
└───────┴─────────────┴───────┴──────────┴────────┘
{ height: 28, width: 2 }
┌───────┬──────────┬─────────────────────┐
│ (idx) │ details  │ amount              │
├───────┼──────────┼─────────────────────┤
│     0 │ " Sh***" │ -322                │
│     1 │ " 40***" │ -300                │
│     2

In [28]:
let fullBankRecords: BankTransaction[] = []
for (let i =0; i< pdfFiles.length; i++) {
  const pdfPath = join(statementsDir, pdfFiles[i]);
  const rawRecords = await extract(pdfPath)
  const bankTransactionRecords = transform(rawRecords)
  fullBankRecords = [...fullBankRecords, ...bankTransactionRecords]
}

const fullDf = pl
.DataFrame(fullBankRecords)

const aggregatedDf = fullDf
.groupBy("details")
.agg(pl.col("amount").sum()).sort(pl.col("amount"), true)
displayTable(maskColumn(fullDf, "details"))
displayTable(maskColumn(aggregatedDf, "details"))

const netWorth2025 = fullDf.select(pl.col("amount").sum().alias("net_worth_2025"))
displayTable(netWorth2025)
console.log("😭😭😭😭 im broke")
// const carsaAnomaly = fullDf.filter(pl.col("details").str.toLowerCase().str.contains("carsa"))
// displayTable(carsaAnomaly)

{ height: 739, width: 4 }
┌───────┬─────────────┬───────┬──────────┬────────┐
│ (idx) │ date        │ type  │ details  │ amount │
├───────┼─────────────┼───────┼──────────┼────────┤
│     0 │ "18 Dec 24" │ ")))" │ " WM***" │ -6.25  │
│     1 │ "18 Dec 24" │ "VIS" │ " am***" │ -4.49  │
│     2 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.91  │
│     3 │ "19 Dec 24" │ "VIS" │ " UB***" │ -8.98  │
│     4 │ "20 Dec 24" │ ")))" │ " GR***" │ -2     │
│     5 │ "20 Dec 24" │ ")))" │ " WM***" │ -9.05  │
│     6 │ "23 Dec 24" │ "VIS" │ " WW***" │ -42.17 │
│     7 │ "23 Dec 24" │ "VIS" │ " am***" │ -5.99  │
│     8 │ "23 Dec 24" │ ")))" │ " NY***" │ -2.2   │
│     9 │ "23 Dec 24" │ ")))" │ " WM***" │ -5     │
└───────┴─────────────┴───────┴──────────┴────────┘
{ height: 273, width: 2 }
┌───────┬──────────┬────────────────────┐
│ (idx) │ details  │ amount             │
├───────┼──────────┼────────────────────┤
│     0 │ " OF***" │ 22859.78           │
│     1 │ " Jo***" │ 10535              │
│     2 │ 